# L21 · Visualización: de "esto funciona" a "esto parece profesional"

| | |
|---|---|
| **Qué vas a aprender** | La diferencia entre un gráfico y un gráfico que alguien se cree |
| **Cuánto tiempo** | 90 min |
| **Datasets** | `ventas_2024_este.csv` |

> **Al terminar vas a poder** decidir qué gráfico toca según la pregunta, y tener un
> tema visual consistente en todo un informe.

In [ ]:
import sys
from pathlib import Path

# JupyterLab arrancado en la raíz del proyecto deja el cwd ahí.
# Si abres el notebook con el cwd en notebooks/, sube un nivel.
ROOT = Path.cwd()
if not (ROOT / "datos").exists() and (ROOT.parent / "datos").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks"))
DATOS = ROOT / "datos"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from herramientas import (cargar, mostrar, ficha, guardar_csv, guardar_excel,
                           guardar_figura, parsear_importes, parsear_fecha,
                           clave_canonica, detectar_alfabeto, auditar_fusiones,
                           resumen_entrega, perfil_memoria, contar_falsos_nulos)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
plt.style.use("seaborn-v0_8-whitegrid")
pd.__version__

In [ ]:
df = pd.read_csv(
    DATOS / "ventas_2024_este.csv", sep=";", decimal=",", encoding="latin-1"
)
df["Fecha Pedido"] = pd.to_datetime(df["Fecha Pedido"], format="%d/%m/%Y")
df["Importe"] = parsear_importes(df["Importe"])
df["Descuento %"] = pd.to_numeric(df["Descuento %"], errors="coerce")
df["mes"] = df["Fecha Pedido"].dt.to_period("M").astype(str)

print("dataset listo:", df.shape)

## El concepto

Un gráfico **no es una decoración de un análisis**: es la respuesta visual a una
pregunta. Antes de elegir el tipo de gráfico, escribe la pregunta:

| La pregunta es... | El gráfico es... |
|---|---|
| ¿Cuál es el MAYOR? | barra **horizontal**, ordenada por valor |
| ¿Cómo EVOLUCIONA? | línea |
| ¿Cómo se DISTRIBUYE? | histograma o caja |
| ¿Se RELACIONAN A y B? | dispersión |
| ¿De qué se COMPONE? | barra apilada (con ≤5 categorías) |

Si no puedes escribir la pregunta en una frase, el gráfico no es el correcto.

## Manos a la obra

### El tema: define tu sistema visual UNA vez

Un informe se ve profesional por **consistencia**, no por sofisticación. Define una
paleta con significado fijo y úsala en todos los gráficos del proyecto.

In [ ]:
# Sistema de diseño del curso. Definido UNA vez, usado en todas partes.
PALETA = {
    "primario":  "#1F4E79",   # azul oscuro: la serie principal
    "secundario":"#E8A33D",   # naranja: la comparación / el foco
    "alerta":    "#C0392B",   # rojo: SOLO para problemas
    "referencia":"#8A94A6",   # gris: contexto, media móvil, comparativa
    "fondo":     "#FFFFFF",
}

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",     # titulo a la izquierda, no centrado
    "axes.labelsize": 10,
    "axes.spines.top": False,        # quita la caja de arriba
    "axes.spines.right": False,      # y la de la derecha
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "-",
    "legend.frameon": False,
    "font.size": 10,
})

print("tema aplicado")

### Gráfico 1 · Evolución → línea

Reglas: título con la unidad, ejes sin cajas innecesarias, y el eje Y empieza en cero
cuando la magnitud es comparable.

In [ ]:
mensual = df.groupby("mes")["Importe"].sum().sort_index() / 1000

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(mensual.index, mensual.values, color=PALETA["primario"], lw=2)
ax.set_title("Ventas mensuales (miles de €)", loc="left")
ax.set_ylabel("Miles de €")
ax.set_ylim(0, mensual.max() * 1.12)
ax.set_xticks(range(len(mensual.index)))
ax.set_xticklabels(mensual.index, rotation=45, ha="right")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

### Gráfico 2 · Comparar → barra **horizontal ordenada**

Dos reglas que la mayoría no cumple: ordenar por valor (no alfabético) y poner los
nombres largos en horizontal, porque en vertical las etiquetas rotan 45° y son
ilegibles.

In [ ]:
por_categoria = df.groupby("Categoria")["Importe"].sum().sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(por_categoria.index, por_categoria.values / 1000, color=PALETA["primario"])
ax.set_title("Facturación por categoría (miles de €)", loc="left")
ax.set_xlabel("Miles de €")
ax.grid(axis="x", alpha=0.3)
ax.set_xlim(0, por_categoria.max() / 1000 * 1.15)

# etiquetar directamente: mejor que una leyenda que obliga a saltar la vista
for y, v in enumerate(por_categoria.values / 1000):
    ax.text(v + por_categoria.max() / 1000 * 0.02, y, f"{v:,.0f}", va="center", fontsize=9)

fig.tight_layout()
plt.show()

### Gráfico 3 · Distribuir → histograma

Un histograma agregado por categoría esconde la variabilidad. Si hay muchas
categorías, usa **small multiples**.

In [ ]:
top4 = df.groupby("Categoria")["Importe"].sum().nlargest(4).index
fig, axes = plt.subplots(2, 2, figsize=(10, 5), sharex=True, sharey=True)

for ax, cat in zip(axes.flat, top4):
    sub = df.loc[df["Categoria"] == cat, "Importe"]
    ax.hist(sub, bins=40, color=PALETA["primario"], alpha=0.85)
    ax.set_title(f"{cat}  (n={len(sub):,})", fontsize=10)
    ax.set_ylim(0, None)
    ax.grid(axis="y", alpha=0.2)

fig.suptitle("Distribución del importe por categoría (€)", x=0.01, ha="left",
             fontsize=13, weight="bold")
fig.tight_layout()
plt.show()

## ⚠️ La trampa

### La trampa que más caro sale: el eje Y truncado

Un gráfico de barras con el eje Y empezando en 80 convierte una diferencia del 2% en
una apariencia del 40%. No es un error de datos: es una **mentira visual**.

In [ ]:
reg = df.groupby("Region")["Importe"].sum().sort_values(ascending=False)
partido = reg.loc[["Norte", "Centro"]]
dif_pct = (partido.iloc[0] - partido.iloc[1]) / partido.iloc[1] * 100
print(f"{partido.index[0]}: {partido.iloc[0]:,.0f} €")
print(f"{partido.index[1]}: {partido.iloc[1]:,.0f} €")
print(f"diferencia real: {dif_pct:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))

# MAL: eje truncado -> la diferencia del 2% parece enorme
axes[0].bar(partido.index, partido.values / 1000, color=PALETA["alerta"])
axes[0].set_ylim(partido.min() / 1000 * 0.99, partido.max() / 1000 * 1.005)
axes[0].set_title(f"TRUNCADO: {dif_pct:+.1f}% parece un abismo", loc="left", color=PALETA["alerta"])
axes[0].grid(axis="y", alpha=0.3)

# BIEN: desde cero -> la diferencia real se ve
axes[1].bar(partido.index, partido.values / 1000, color=PALETA["primario"])
axes[1].set_ylim(0, partido.max() / 1000 * 1.15)
axes[1].set_title("DESDE CERO: la diferencia real", loc="left")
axes[1].grid(axis="y", alpha=0.3)

for ax in axes:
    ax.set_ylabel("Miles de €")

fig.tight_layout()
plt.show()

### Y la lista de prohibiciones

- gráficos 3D: distorsionan las longitudes, y todos repiten los valores a la derecha
- sombras y bordes decorativos: no aportan información
- más de 5 colores: si necesitas más, estás mezclando dos gráficos
- leyenda cuando puedes etiquetar directamente
- ejes Y truncados en gráficos de barras
- `plt.` suelto después de haber creado `ax`: el 95% de los errores de matplotlib
  vienen de mezclar las dos APIs

In [ ]:
# plt.title NO borra las barras: pinta el título en los ejes activos.
fig, ax = plt.subplots(figsize=(6, 2.5))
ax.bar(["A", "B", "C"], [3, 5, 2], color=PALETA["primario"])
plt.title("Este titulo cae en la figura de arriba, que sigue siendo la activa")

# El lio de verdad: otro plt.subplots() cambia la figura activa.
# El plt.title de despues ya no toca el grafico de las barras.
fig2, ax2 = plt.subplots(figsize=(6, 2.2))
plt.title("Este titulo ha caido en la figura nueva")
fig.tight_layout()
fig2.tight_layout()
plt.show()

print("por eso: guarda fig, ax y llama a ax.set_title, no a plt.title")

## 🧪 Tu turno

- **Ejercicio 1** — Reescribe el gráfico de barras por categoría usando un **boxplot**
  o un violín, y decide cuál de los dos informa mejor. Justifica en una frase.
- **Ejercicio 2** — La paleta `PALETA` tiene 5 colores. ¿En qué gráfico tuyo actual
  usarías el color de alerta, y por qué solo ahí?
- **Ejercicio 3** — Exporta tu gráfico de ventas mensuales a los **tres** formatos
  (PNG, SVG, PDF) y abre el SVG en un navegador: ¿pesa más o menos que el PNG? ¿Por qué?

## 📦 Workbook integrador: dashboard de KPI

**Contexto.** El comité de dirección de un retailer pide un panel con 4 indicadores
para la reunión mensual. No quieren 40 gráficos: quieren **4** que respondan a 4
preguntas concretas, con el mismo estilo visual.

**Tu trabajo.** Un PNG de 2×2 con:

1. **Evolución** — ventas mensuales (línea) con la media del periodo como referencia.
2. **Comparación** — top 5 categorías por facturación (barra horizontal ordenada).
3. **Distribución** — histograma de importes por pedido, con la mediana marcada.
4. **Composición** — reparto por canal de venta, ordenado y con porcentajes.

**Requisitos duros de calidad:**

- el mismo `PALETA` y los mismos `rcParams` en los cuatro;
- todos con `fig, ax = plt.subplots()` y `tight_layout()`;
- títulos con la unidad, alineados a la izquierda;
- **máximo 3 colores** en todo el dashboard;
- nada de leyenda donde puedas etiquetar;
- exportado a **PNG (dpi=200) y SVG**.

Además, exporta un **PNG "antes/después"** con los mismos datos hechos mal (eje
truncado, 13 colores, 3D) y bien, para que el comité vea por qué importa.

In [ ]:
# --- 1. Evolución ---
mensual = df.groupby("mes")["Importe"].sum().sort_index() / 1000
media = mensual.mean()

# --- 2. Comparación ---
categorias = df.groupby("Categoria")["Importe"].sum().nlargest(5).sort_values() / 1000

# --- 3. Distribución ---
mediana = df["Importe"].median() / 1000

# --- 4. Composición ---
canales = (df.groupby("Canal")["Importe"].sum().sort_values(ascending=False) / 1000)

print("4 indicadores calculados. montando el panel 2x2...")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7.5))
fig.suptitle("Panel mensual · Ventas por canal, categoría y periodo",
             x=0.01, ha="left", fontsize=15, weight="bold")

# 1. evolución
ax = axes[0, 0]
ax.plot(mensual.index, mensual.values, color=PALETA["primario"], lw=2)
ax.axhline(media, color=PALETA["referencia"], ls="--", lw=1.2)
ax.text(len(mensual) - 1, media, f"  media {media:,.0f}k €", va="center",
        fontsize=9, color=PALETA["referencia"])
ax.set_title("1 · Evolución mensual (miles de €)", loc="left")
ax.set_ylim(0, mensual.max() * 1.15)
ax.set_xticks(range(len(mensual.index)))
ax.set_xticklabels(mensual.index, rotation=45, ha="right")
ax.grid(axis="y", alpha=0.3)

# 2. comparación
ax = axes[0, 1]
ax.barh(categorias.index, categorias.values, color=PALETA["primario"])
ax.set_title("2 · Top 5 categorías (miles de €)", loc="left")
ax.set_xlim(0, categorias.max() * 1.18)
ax.grid(axis="x", alpha=0.3)
for y, v in enumerate(categorias.values):
    ax.text(v + categorias.max() * 0.02, y, f"{v:,.0f}", va="center", fontsize=9)

# 3. distribución
ax = axes[1, 0]
ax.hist(df["Importe"] / 1000, bins=45, color=PALETA["primario"], alpha=0.85)
ax.axvline(mediana, color=PALETA["secundario"], lw=2)
ax.text(mediana, ax.get_ylim()[1] * 0.92, f" mediana {mediana:,.1f}k €",
        fontsize=9, color=PALETA["secundario"], weight="bold")
ax.set_title("3 · Distribución del importe por pedido (miles de €)", loc="left")
ax.set_xlabel("Miles de €")
ax.grid(axis="y", alpha=0.3)

# 4. composición
# El color SOLO se usa si significa algo. Aquí los canales están casi empatados,
# así que colorearlos sería decoración: highlight real = "el mejor", gris = "el resto".
ax = axes[1, 1]
colores = [PALETA["primario"] if v == canales.max() else PALETA["referencia"]
           for v in canales.values]
ax.bar(canales.index, canales.values, color=colores)
ax.set_title("4 · Reparto por canal (miles de €)", loc="left")
ax.set_ylabel("Miles de €")
ax.set_xticks(range(len(canales.index)))
ax.set_xticklabels(canales.index, rotation=30, ha="right")
ax.set_ylim(0, canales.max() * 1.2)
ax.grid(axis="y", alpha=0.3)
for x, v in enumerate(canales.values):
    pct = v / canales.sum() * 100
    ax.text(x, v + canales.max() * 0.02, f"{pct:.0f}%", ha="center", fontsize=9)

fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
rutas = guardar_figura(fig, "L21_dashboard_kpi", formatos=("png", "svg", "pdf"),
                      dpi=200)

In [ ]:
# el "antes y después" que el comité pidió
fig2, ax2 = plt.subplots(figsize=(7, 3.6))
ax2.bar(partido.index, partido.values / 1000, color=PALETA["alerta"])
ax2.set_ylim(partido.min() / 1000 * 0.99, partido.max() / 1000 * 1.005)
ax2.set_title(f"MAL: eje truncado. La diferencia real es {dif_pct:+.1f}%",
              loc="left", color=PALETA["alerta"], weight="bold")
ax2.set_ylabel("Miles de €")
ax2.grid(axis="y", alpha=0.3)
fig2.tight_layout()

ruta_mal = guardar_figura(fig2, "L21_ejemplo_mal_grafico", formatos=("png",))
plt.show()

resumen_entrega(*rutas, ruta_mal)

### Criterios de aceptación

1. El dashboard se guarda en `salida/L21_dashboard_kpi.png` a **dpi=200**, y además en
   `.svg` y `.pdf`.
2. Los 4 paneles usan la **misma** `PALETA` y los mismos `rcParams`. Compruébalo
   recortando un panel: tiene que encajar con los otros tres.
3. Ningún panel tiene leyenda; los datos están etiquetados directamente.
4. El eje Y de los paneles de barras **empieza en cero**.
5. No hay más de 3 colores distintos en todo el dashboard.
6. `L21_ejemplo_mal_grafico.png` existe y muestra el eje truncado.
7. En el panel 4, los porcentajes suman entre 99 y 101 (redondeo).